# Chapter 18 — RAG Production Patterns: The Gap Between Demo and Reality

**Volume 2: Practical Applications — AI for Networking and Security Engineers**

The demo worked perfectly. Then you put it in front of 50 engineers and:
- Response time is 8 seconds
- You are spending $300/month on API calls
- The same question is processed from scratch every time
- A new document was added and nobody knows if it was indexed

> *In Simple Words: Building a RAG demo is like setting up a test router with static routes.
> Building a production RAG system is like running BGP with route policies, redundancy,
> monitoring, and change management. Same protocol — completely different operational requirements.*

---
**What you will build — 5 hands-on demos:**

| # | Demo | Key concept |
|---|------|-------------|
| 1 | Semantic Cache | Same question, different words — serve cached answer |
| 2 | LLM Cascade | Haiku first (cheap), escalate to Sonnet only if needed |
| 3 | RAG Evaluation (LLM-as-Judge) | Faithfulness, Context Recall, Answer Relevancy |
| 4 | Streaming + Latency Monitor | TTFT, tokens/sec, per-query cost tracking |
| 5 | Full Production Pipeline | All patterns combined: cache → cascade → stream → evaluate |

> **Prerequisite:** `!pip install anthropic scikit-learn numpy` (run install cell first)


In [ ]:
!pip install -q anthropic scikit-learn numpy

---
## Demo 1 — Semantic Cache: Stop Paying for the Same Question Twice

**The cost problem:**
100 engineers × 20 queries/day = 2,000 queries/day.
At $0.003/1K input + $0.015/1K output ≈ **$1,170/month**.
And 30–40% of those queries are semantically identical.

"What is the MTU on the uplink?" and "What MTU do we use on the backbone?"
— different strings, same question. Traditional key-value cache misses this.

**Semantic cache** compares query *embeddings*, not strings:
```
New query arrives
       │
       ▼  embed the query
Compute similarity vs ALL cached query embeddings
       │
   score ≥ 0.92? ── YES ──▶ Return cached answer  (cost = $0)
       │
      NO
       │
       ▼  process normally (retrieve → generate)
Cache the new query + answer pair
```

**The threshold problem:** Too low (0.80) → wrong answers for different questions.
Too high (0.98) → almost no cache hits. Sweet spot for networking: **0.92–0.95**.

> *Networking analogy: BGP route caching in the FIB.
> Don't look up the route every time — cache it. Invalidate on topology change.*


In [ ]:
# Demo 1: Semantic cache with TF-IDF embeddings + TTL eviction
import time
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ── Semantic Cache ────────────────────────────────────────────────────────────
class SemanticCache:
    """
    Cache that matches semantically similar queries, not just identical strings.
    Uses TF-IDF embeddings + cosine similarity for matching.
    LRU eviction with TTL to prevent stale answers.
    """

    def __init__(self, threshold: float = 0.88, max_size: int = 100, ttl_seconds: int = 3600):
        self.threshold   = threshold      # similarity threshold for cache hit
        self.max_size    = max_size       # maximum cached entries
        self.ttl         = ttl_seconds    # time-to-live in seconds
        self.cache       = {}             # {query_text: {answer, timestamp, hits}}
        self.vectorizer  = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
        self._fitted     = False
        self.stats       = {"hits": 0, "misses": 0, "evictions": 0}

    def _rebuild_index(self):
        """Rebuild TF-IDF index from all cached queries."""
        if not self.cache:
            self._fitted = False
            return
        queries = list(self.cache.keys())
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
        self._matrix   = self.vectorizer.fit_transform(queries)
        self._queries  = queries
        self._fitted   = True

    def _is_expired(self, entry: dict) -> bool:
        return time.time() - entry["timestamp"] > self.ttl

    def lookup(self, query: str) -> tuple:
        """
        Check if a semantically similar query is cached.
        Returns (answer, similarity_score) or (None, 0.0).
        """
        # Remove expired entries first
        expired = [q for q, e in self.cache.items() if self._is_expired(e)]
        for q in expired:
            del self.cache[q]
            self.stats["evictions"] += 1
        if expired:
            self._rebuild_index()

        if not self._fitted or not self.cache:
            self.stats["misses"] += 1
            return None, 0.0

        try:
            q_vec  = self.vectorizer.transform([query])
            scores = cosine_similarity(q_vec, self._matrix).flatten()
            best_idx   = int(np.argmax(scores))
            best_score = float(scores[best_idx])

            if best_score >= self.threshold:
                matched_query = self._queries[best_idx]
                self.cache[matched_query]["hits"] += 1
                self.cache[matched_query]["timestamp"] = time.time()   # LRU refresh
                self.stats["hits"] += 1
                return self.cache[matched_query]["answer"], best_score
        except Exception:
            pass

        self.stats["misses"] += 1
        return None, 0.0

    def store(self, query: str, answer: str):
        """Add a query-answer pair to the cache."""
        # Evict LRU if at capacity
        if len(self.cache) >= self.max_size:
            oldest = min(self.cache, key=lambda q: self.cache[q]["timestamp"])
            del self.cache[oldest]
            self.stats["evictions"] += 1

        self.cache[query] = {
            "answer":    answer,
            "timestamp": time.time(),
            "hits":      0,
        }
        self._rebuild_index()

    def hit_rate(self) -> float:
        total = self.stats["hits"] + self.stats["misses"]
        return self.stats["hits"] / total if total > 0 else 0.0

    def report(self):
        print(f"  Cache stats: {self.stats['hits']} hits | "
              f"{self.stats['misses']} misses | "
              f"{self.stats['evictions']} evictions | "
              f"hit rate: {self.hit_rate():.0%}")

# ── Simulate a RAG pipeline (no real API needed for cache demo) ───────────────
QUESTION_ANSWER_PAIRS = [
    ("What is the BGP AS number for our network?",
     "Our network uses AS number 65001. This is configured on all core routers."),
    ("What MTU is configured on the uplink interfaces?",
     "Uplink interfaces use MTU 9000 (jumbo frames) to support large payloads."),
    ("How do I configure OSPF area 0 authentication?",
     "Use MD5 authentication: area 0 authentication message-digest under router ospf."),
    ("What is the BGP hold timer setting?",
     "BGP hold timer is 90s, keepalive is 30s. Configured globally on all peers."),
    ("Which NTP server do we use?",
     "Primary NTP: 10.0.0.1 (stratum 2). Backup: 10.0.0.2. Configured on all devices."),
]

cache = SemanticCache(threshold=0.88, ttl_seconds=3600)

# Pre-warm the cache with known Q&A pairs
print("Pre-warming cache with 5 common Q&A pairs...
")
for q, a in QUESTION_ANSWER_PAIRS:
    cache.store(q, a)

# ── Test with semantically similar (but not identical) queries ─────────────────
test_queries = [
    ("What AS number does our network use?",           "paraphrase of Q1"),
    ("What is the jumbo frame MTU on backbone links?", "paraphrase of Q2"),
    ("OSPF area 0 auth configuration",                 "shorter version of Q3"),
    ("What is the BGP timer configuration?",           "near-synonym of Q4"),
    ("How do I configure IS-IS on a loopback?",        "unrelated — should miss"),
]

print("="*65)
print("  SEMANTIC CACHE — LIVE TEST")
print(f"  Similarity threshold: {cache.threshold}")
print("="*65)

for query, note in test_queries:
    t0 = time.time()
    answer, score = cache.lookup(query)
    elapsed = (time.time() - t0) * 1000   # ms

    if answer:
        print(f"
  ✅ HIT  [{score:.3f}] {query}")
        print(f"       Note:   {note}")
        print(f"       Answer: {answer[:70]}...")
        print(f"       Time:   {elapsed:.2f}ms (no API call needed)")
    else:
        print(f"
  ❌ MISS [{score:.3f}] {query}")
        print(f"       Note:  {note}")
        print(f"       → Would call the LLM API and cache the result")

print()
cache.report()

# ── Cost savings estimate ─────────────────────────────────────────────────────
queries_per_day = 200
hit_rate = cache.hit_rate()
cost_per_query = 0.006   # $0.006 avg cost (context + output)
daily_savings  = queries_per_day * hit_rate * cost_per_query
monthly_savings = daily_savings * 30
print(f"
  Estimated savings at {hit_rate:.0%} hit rate:")
print(f"    Daily:   ${daily_savings:.2f}")
print(f"    Monthly: ${monthly_savings:.2f}")


---
## Demo 2 — LLM Cascade: Small Model First, Escalate Only When Needed

**The economics:**
- Claude Haiku: ~$0.001 per query (fast, cheap)
- Claude Sonnet: ~$0.008 per query (8× more expensive, much stronger)

If 65% of queries can be handled by Haiku correctly, you save 65% of LLM costs.
The complex 35% still get full Sonnet quality.

**The cascade flow:**
```
Query arrives
     │
     ▼  Step 1: Try Haiku
Haiku generates answer + self-assessed confidence (1-10)
     │
 confidence ≥ 7? ── YES ──▶ Return Haiku answer   (cheap)
     │
    NO
     │
     ▼  Step 2: Escalate to Sonnet
Sonnet generates high-quality answer
     │
     ▼  Return Sonnet answer
```

**What makes a query "easy" for Haiku?**
- Direct factual lookups with the answer clearly in the context
- Simple yes/no questions about configurations
- Single-hop reasoning ("what is X configured as?")

**What needs Sonnet?**
- Multi-hop reasoning ("given X and Y, what is the impact on Z?")
- Complex trade-off analysis
- Ambiguous contexts requiring synthesis

> *In Simple Words: Tiered support — Level 1 resolves simple tickets fast and cheap.
> Complex tickets escalate to Level 2. Same principle, different models.*


In [ ]:
# Demo 2: LLM cascade — Haiku first, escalate to Sonnet if confidence is low
import os
from anthropic import Anthropic
import json
import time

try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    import getpass
    if not os.environ.get('ANTHROPIC_API_KEY'):
        os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Enter your Anthropic API key: ')
client = Anthropic()
HAIKU  = "claude-haiku-4-5-20251001"
SONNET = "claude-sonnet-4-20250514"

# ── Simulated knowledge base context ─────────────────────────────────────────
CONTEXT = """Network Configuration Reference — Datacenter Core v2.4

BGP Configuration:
- AS number: 65001
- BGP hold timer: 90 seconds, keepalive: 30 seconds
- Neighbor 203.0.113.2 (AS 65000): ISP uplink peer
- Neighbor 10.1.1.2 (AS 65001): iBGP peer (route reflector)
- Maximum-prefix limit: 1000 prefixes per neighbor

OSPF Configuration:
- Process ID: 1, Router ID: 10.0.0.1
- Area 0: backbone, authentication MD5
- Hello interval: 10s, Dead interval: 40s (broadcast links)
- SPF delay: 50ms initial, 200ms minimum, 5000ms maximum

Interface MTU:
- Core uplinks: 9000 bytes (jumbo frames)
- Access layer: 1500 bytes (standard)
- Loopbacks: 65535 bytes (software only)
"""

def cascade_query(question: str, context: str, confidence_threshold: int = 7) -> dict:
    """
    Try Haiku first. If it self-reports low confidence, escalate to Sonnet.
    Returns: {answer, model_used, confidence, escalated, latency_ms}
    """

    system_prompt = (
        "You are a network operations assistant. Answer using ONLY the provided context. "
        "After your answer, on a new line write exactly: CONFIDENCE: X "
        "where X is 1-10 (10 = completely certain from context, 1 = guessing). "
        "Be honest about confidence."
    )

    # ── Stage 1: Try Haiku ────────────────────────────────────────────────────
    t0 = time.time()
    haiku_resp = client.messages.create(
        model=HAIKU,
        max_tokens=200,
        system=system_prompt,
        messages=[{"role": "user", "content": f"Context:
{context}

Question: {question}"}],
    )
    haiku_time = (time.time() - t0) * 1000
    haiku_raw  = haiku_resp.content[0].text.strip()

    # Extract confidence score
    confidence = 5   # default
    answer_text = haiku_raw
    if "CONFIDENCE:" in haiku_raw:
        parts = haiku_raw.rsplit("CONFIDENCE:", 1)
        answer_text = parts[0].strip()
        try:
            confidence = int(parts[1].strip().split()[0])
        except (ValueError, IndexError):
            confidence = 5

    # ── Stage 2: Escalate to Sonnet if confidence too low ─────────────────────
    if confidence >= confidence_threshold:
        return {
            "answer":      answer_text,
            "model":       "haiku",
            "confidence":  confidence,
            "escalated":   False,
            "latency_ms":  haiku_time,
            "cost_est":    0.001,
        }

    # Escalate
    t0 = time.time()
    sonnet_resp = client.messages.create(
        model=SONNET,
        max_tokens=300,
        system=system_prompt,
        messages=[{"role": "user", "content": f"Context:
{context}

Question: {question}"}],
    )
    sonnet_time = (time.time() - t0) * 1000
    sonnet_raw  = sonnet_resp.content[0].text.strip()
    sonnet_ans  = sonnet_raw.rsplit("CONFIDENCE:", 1)[0].strip() if "CONFIDENCE:" in sonnet_raw else sonnet_raw

    try:
        sonnet_conf = int(sonnet_raw.rsplit("CONFIDENCE:", 1)[1].strip().split()[0])
    except Exception:
        sonnet_conf = 8

    return {
        "answer":      sonnet_ans,
        "model":       "sonnet",
        "confidence":  sonnet_conf,
        "escalated":   True,
        "latency_ms":  haiku_time + sonnet_time,
        "cost_est":    0.009,   # Haiku + Sonnet
    }

# ── Test questions ranging from simple to complex ─────────────────────────────
questions = [
    ("What is our BGP AS number?",                             "Simple factual lookup"),
    ("What is the OSPF hello interval on broadcast links?",    "Direct context lookup"),
    ("What MTU do we use on core uplinks?",                    "Direct factual"),
    ("Why would OSPF adjacency fail if MTU differs between peers?", "Reasoning beyond context"),
    ("Compare the trade-offs of our BGP timer settings vs faster detection", "Complex analysis"),
]

print("="*65)
print("  LLM CASCADE DEMO")
print(f"  Confidence threshold for escalation: 7/10")
print("="*65)

total_cost = 0
haiku_count = 0
sonnet_count = 0

for question, note in questions:
    result = cascade_query(question, CONTEXT, confidence_threshold=7)
    total_cost += result["cost_est"]

    icon = "⬆️ ESCALATED" if result["escalated"] else "✅ HAIKU"
    if result["escalated"]:
        sonnet_count += 1
    else:
        haiku_count += 1

    print(f"
  Q: {question}")
    print(f"     Note: {note}")
    print(f"     {icon}  | Confidence: {result['confidence']}/10 | {result['latency_ms']:.0f}ms | ${result['cost_est']:.3f}")
    print(f"     Answer: {result['answer'][:100]}...")

print(f"
{'='*65}")
print(f"  COST SUMMARY:")
print(f"    Haiku answers (cheap):   {haiku_count}/{len(questions)}")
print(f"    Sonnet escalations:      {sonnet_count}/{len(questions)}")
print(f"    Total cost (5 queries):  ${total_cost:.3f}")

baseline_sonnet = len(questions) * 0.008
savings = baseline_sonnet - total_cost
print(f"    vs all-Sonnet baseline:  ${baseline_sonnet:.3f}")
print(f"    Savings:                 ${savings:.3f} ({savings/baseline_sonnet:.0%})")


---
## Demo 3 — RAG Evaluation: Measuring Quality Automatically

How do you know your RAG is giving good answers at scale?
You can't manually review 2,000 queries/day.

**The RAGAS framework — 4 core metrics:**

| Metric | What it measures | Formula |
|--------|-----------------|---------|
| **Faithfulness** | Are all claims supported by the retrieved context? | supported_claims / total_claims |
| **Context Precision** | Are relevant chunks ranked above irrelevant ones? | relevant_at_K / K |
| **Context Recall** | Did retrieval find all the needed information? | found_needed / total_needed |
| **Answer Relevancy** | Does the answer actually address the question? | LLM judge 0–1 |

**Faithfulness is the most important.**
A faithfulness score < 0.8 means your system is hallucinating —
generating plausible-sounding but ungrounded statements.

In networking, this is dangerous:
"The recommended MTU for backbone links is 9000 bytes" — if that's not in YOUR documentation,
it came from the LLM's training data and might be wrong for your specific setup.

> *LLM-as-Judge: Use Claude to evaluate Claude. Score each metric 0–1.
> This is the RAGAS approach applied without any extra libraries.*


In [ ]:
from anthropic import Anthropic
# Demo 3: RAG evaluation — 4 RAGAS-style metrics using LLM-as-Judge
import json
import re

client = Anthropic()
HAIKU = "claude-haiku-4-5-20251001"

def llm_judge(prompt: str, max_tokens: int = 150) -> str:
    resp = client.messages.create(
        model=HAIKU, max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.content[0].text.strip()

def extract_score(text: str) -> float:
    """Extract a 0.0–1.0 score from LLM response."""
    match = re.search(r'(?:score|rating|result)[:\s]+([0-9.]+)', text, re.IGNORECASE)
    if not match:
        match = re.search(r'([0-9]\.[0-9]+|[01]\.0|0|1)', text)
    try:
        val = float(match.group(1)) if match else 0.5
        return min(1.0, max(0.0, val if val <= 1.0 else val / 10.0))
    except Exception:
        return 0.5

# ── Metric 1: Faithfulness ────────────────────────────────────────────────────
def faithfulness(question: str, context: str, answer: str) -> dict:
    """
    Faithfulness = claims in answer supported by context / total claims in answer.
    """
    prompt = (
        f"Evaluate FAITHFULNESS: are all claims in the answer supported by the context?

"
        f"Context: {context}

Question: {question}

Answer: {answer}

"
        f"Count the claims in the answer. Count how many are directly supported by the context. "
        f"Respond with JSON: {{"supported": N, "total": N, "score": 0.0-1.0, "reason": "..."}}"
    )
    raw = llm_judge(prompt, max_tokens=200)
    try:
        match = re.search(r'\{[\s\S]*\}', raw)
        result = json.loads(match.group()) if match else {}
        return {
            "metric": "faithfulness",
            "score":  float(result.get("score", 0.5)),
            "detail": result.get("reason", raw[:100]),
        }
    except Exception:
        return {"metric": "faithfulness", "score": extract_score(raw), "detail": raw[:100]}

# ── Metric 2: Context Relevance ───────────────────────────────────────────────
def context_relevance(question: str, context_chunks: list) -> dict:
    """
    Context Precision = what fraction of retrieved chunks are actually relevant?
    """
    chunks_text = "
".join(f"[Chunk {i+1}] {c}" for i, c in enumerate(context_chunks))
    prompt = (
        f"Evaluate CONTEXT RELEVANCE: for each retrieved chunk, is it relevant to the question?

"
        f"Question: {question}

Retrieved chunks:
{chunks_text}

"
        f"For each chunk, say relevant or not_relevant. "
        f"Respond with JSON: {{"relevant_count": N, "total": N, "score": 0.0-1.0}}"
    )
    raw = llm_judge(prompt, max_tokens=150)
    try:
        match = re.search(r'\{[\s\S]*\}', raw)
        result = json.loads(match.group()) if match else {}
        return {
            "metric": "context_relevance",
            "score":  float(result.get("score", 0.5)),
            "detail": f"{result.get('relevant_count','?')}/{result.get('total','?')} chunks relevant",
        }
    except Exception:
        return {"metric": "context_relevance", "score": extract_score(raw), "detail": raw[:100]}

# ── Metric 3: Answer Relevancy ────────────────────────────────────────────────
def answer_relevancy(question: str, answer: str) -> dict:
    """
    Does the answer actually address what was asked?
    """
    prompt = (
        f"Evaluate ANSWER RELEVANCY: does the answer directly address the question?

"
        f"Question: {question}

Answer: {answer}

"
        f"Score 0.0 (completely off-topic) to 1.0 (directly and completely answers the question). "
        f"Respond with JSON: {{"score": 0.0-1.0, "reason": "..."}}"
    )
    raw = llm_judge(prompt, max_tokens=100)
    try:
        match = re.search(r'\{[\s\S]*\}', raw)
        result = json.loads(match.group()) if match else {}
        return {
            "metric": "answer_relevancy",
            "score":  float(result.get("score", 0.5)),
            "detail": result.get("reason", raw[:100]),
        }
    except Exception:
        return {"metric": "answer_relevancy", "score": extract_score(raw), "detail": raw[:100]}

# ── Evaluate 3 test cases ─────────────────────────────────────────────────────
CONTEXT_GOOD = (
    "BGP uses AS number 65001 on all core routers. "
    "BGP hold timer is 90 seconds, keepalive is 30 seconds. "
    "Neighbor 203.0.113.2 (AS 65000) is the ISP uplink peer."
)
CONTEXT_NOISY = (
    "NTP server 10.0.0.1 is the primary stratum-2 source. "
    "OSPF area 0 uses MD5 authentication. "
    "BGP hold timer is 90 seconds, keepalive is 30 seconds."   # only partially relevant
)

test_cases = [
    {
        "label":   "Good answer (grounded in context)",
        "question": "What is the BGP hold timer and keepalive interval?",
        "context":  CONTEXT_GOOD,
        "chunks":  [
            "BGP hold timer is 90 seconds, keepalive is 30 seconds.",
            "Neighbor 203.0.113.2 (AS 65000) is the ISP uplink peer.",
        ],
        "answer":   "The BGP hold timer is 90 seconds and the keepalive interval is 30 seconds.",
    },
    {
        "label":   "Hallucinated answer (not in context)",
        "question": "What is the BGP hold timer and keepalive interval?",
        "context":  CONTEXT_GOOD,
        "chunks":  [
            "BGP hold timer is 90 seconds, keepalive is 30 seconds.",
        ],
        "answer":   "The BGP hold timer is 90 seconds and keepalive is 30 seconds. BFD is also enabled with 300ms intervals for sub-second detection.",
    },
    {
        "label":   "Irrelevant answer",
        "question": "What is the BGP hold timer and keepalive interval?",
        "context":  CONTEXT_NOISY,
        "chunks":  ["NTP server 10.0.0.1 is the primary stratum-2 source."],
        "answer":   "The NTP server is 10.0.0.1 with stratum 2.",
    },
]

print("="*65)
print("  RAG EVALUATION — LLM-AS-JUDGE (RAGAS-STYLE METRICS)")
print("="*65)

for tc in test_cases:
    print(f"
  ── {tc['label']} ──────────────────────────────")
    print(f"  Q: {tc['question']}")
    print(f"  A: {tc['answer'][:80]}...")

    f_score  = faithfulness(tc["question"], tc["context"], tc["answer"])
    cr_score = context_relevance(tc["question"], tc["chunks"])
    ar_score = answer_relevancy(tc["question"], tc["answer"])

    for metric in [f_score, cr_score, ar_score]:
        score = metric["score"]
        bar   = "█" * int(score * 10) + "░" * (10 - int(score * 10))
        icon  = "✅" if score >= 0.7 else ("⚠️" if score >= 0.4 else "❌")
        print(f"  {icon} {metric['metric']:20} [{bar}] {score:.2f}  {metric['detail'][:50]}")

    overall = (f_score["score"] + cr_score["score"] + ar_score["score"]) / 3
    print(f"     Overall RAG score: {overall:.2f}")


---
## Demo 4 — Streaming Responses + Latency and Cost Monitoring

**Why streaming matters in production:**
Without streaming: user waits 6–8 seconds, sees nothing, then gets everything at once.
With streaming: user sees the first token in <1 second, reads as it arrives.

The perceived latency drops dramatically even when total latency is the same.

**Key metrics to track:**

| Metric | What it measures | Target |
|--------|-----------------|--------|
| **TTFT** | Time To First Token — how fast does the system respond? | < 1.5s |
| **Throughput** | Tokens per second after TTFT | > 30 tok/s |
| **Total latency** | Full response time end-to-end | < 8s |
| **Cost per query** | Input tokens × price + output tokens × price | < $0.01 |

**Cost formula:**
```
cost = (input_tokens × $0.00025 + output_tokens × $0.00125) / 1000
```
(Haiku pricing — Sonnet is ~10× more expensive per token)

> *Networking analogy: TTFT = ping RTT (how fast does the first bit arrive?).
> Throughput = bandwidth (how fast does the rest stream?).
> You want low latency AND high throughput.*


In [ ]:
from anthropic import Anthropic
# Demo 4: Streaming responses with TTFT, throughput, and cost tracking
import time

client = Anthropic()
HAIKU  = "claude-haiku-4-5-20251001"
SONNET = "claude-sonnet-4-20250514"

# Claude API pricing (per 1M tokens, Feb 2025)
PRICING = {
    "claude-haiku-4-5-20251001":  {"input": 0.80,  "output": 4.00},    # per 1M tokens
    "claude-sonnet-4-20250514":          {"input": 3.00,  "output": 15.00},   # per 1M tokens
}

class QueryMonitor:
    """Track per-query performance metrics."""

    def __init__(self):
        self.history = []

    def run_streaming_query(self, model: str, system: str, question: str, verbose: bool = True) -> dict:
        """Execute a streaming query and collect all performance metrics."""
        t_start      = time.time()
        t_first_token = None
        tokens_out   = 0
        full_response = ""

        if verbose:
            print(f"
  Model: {model.split('-')[1].upper()}")
            print(f"  Q: {question}")
            print(f"  A: ", end="", flush=True)

        with client.messages.stream(
            model=model,
            max_tokens=300,
            system=system,
            messages=[{"role": "user", "content": question}],
        ) as stream:
            for text in stream.text_stream:
                if t_first_token is None:
                    t_first_token = time.time()
                full_response += text
                tokens_out += len(text.split())
                if verbose:
                    print(text, end="", flush=True)

        t_end = time.time()

        ttft          = (t_first_token - t_start) if t_first_token else t_end - t_start
        total_latency = t_end - t_start
        throughput    = tokens_out / (t_end - t_first_token) if t_first_token and t_end > t_first_token else 0

        # Estimate cost (rough: input tokens ≈ len(question.split())*1.3)
        input_tokens  = int(len(question.split()) * 1.3 + len(system.split()) * 1.3)
        price         = PRICING.get(model, {"input": 3.0, "output": 15.0})
        cost          = (input_tokens * price["input"] + tokens_out * price["output"]) / 1_000_000

        record = {
            "model":        model.split("-")[1],
            "question":     question[:40],
            "ttft_s":       round(ttft, 2),
            "total_s":      round(total_latency, 2),
            "throughput":   round(throughput, 1),
            "output_tokens": tokens_out,
            "cost_usd":     round(cost, 5),
        }
        self.history.append(record)
        if verbose:
            print(f"
")
        return record

    def print_metrics(self, record: dict):
        ttft_icon = "✅" if record["ttft_s"] < 1.5 else "⚠️"
        thpt_icon = "✅" if record["throughput"] > 20 else "⚠️"
        print(f"  {ttft_icon} TTFT:        {record['ttft_s']:.2f}s")
        print(f"  {thpt_icon} Throughput:  {record['throughput']:.1f} tokens/sec")
        print(f"     Total time:  {record['total_s']:.2f}s")
        print(f"     Cost:        ${record['cost_usd']:.5f}")

    def summary(self):
        if not self.history:
            return
        print(f"
{'='*65}")
        print(f"  QUERY PERFORMANCE SUMMARY ({len(self.history)} queries)")
        print(f"{'='*65}")
        print(f"  {'Model':10} {'TTFT':>8} {'Total':>8} {'Tok/s':>8} {'Cost':>10}")
        print(f"  {'-'*10} {'-'*8} {'-'*8} {'-'*8} {'-'*10}")
        for r in self.history:
            print(f"  {r['model']:10} {r['ttft_s']:>7.2f}s {r['total_s']:>7.2f}s "
                  f"{r['throughput']:>7.1f} ${r['cost_usd']:>9.5f}")
        total_cost = sum(r["cost_usd"] for r in self.history)
        print(f"
  Total cost for {len(self.history)} queries: ${total_cost:.4f}")
        daily_est  = total_cost / len(self.history) * 2000
        print(f"  Estimated daily cost (2000 queries/day): ${daily_est:.2f}")


# ── Run streaming queries on both models ──────────────────────────────────────
SYSTEM = (
    "You are a network operations assistant. "
    "Answer concisely using your knowledge about BGP, OSPF, and network troubleshooting."
)

monitor = QueryMonitor()
questions = [
    ("What is the default BGP hold timer?",          HAIKU),
    ("Explain why OSPF EXSTART state can be caused by MTU mismatch", SONNET),
    ("What are the BGP notification codes?",         HAIKU),
]

print("="*65)
print("  STREAMING + LATENCY MONITORING DEMO")
print("="*65)

for question, model in questions:
    record = monitor.run_streaming_query(model, SYSTEM, question, verbose=True)
    monitor.print_metrics(record)

monitor.summary()


---
## Demo 5 — Full Production Pipeline: All Patterns Combined

This assembles every production pattern into one complete pipeline:

```
Query arrives
     │
     ▼ 1. SEMANTIC CACHE CHECK
 Cache hit? ── YES ──▶ Return cached answer instantly   ($0 cost)
     │
    NO
     │
     ▼ 2. HYBRID RETRIEVAL (BM25 + TF-IDF + RRF)
 Top-3 relevant chunks retrieved
     │
     ▼ 3. LLM CASCADE (Haiku → Sonnet if needed)
 Generate answer + stream to user
     │
     ▼ 4. CACHE STORE (cache query + answer for next time)
 Store with TTL
     │
     ▼ 5. ASYNC EVALUATION (non-blocking)
 Faithfulness + Relevancy scores logged
     │
     ▼ 6. METRICS RECORDED
 TTFT, cost, cache hit/miss logged for monitoring
```

**The operational loop:**
Every query improves the system — cache fills up, metrics accumulate,
you can see which queries have low faithfulness scores and fix the knowledge base.

> *This is the BGP convergence analogy: each query is a BGP update.
> Over time, the system converges to optimal state — high cache hit rate,
> low average cost, high faithfulness scores.*


In [ ]:
from anthropic import Anthropic
# Demo 5: Full production pipeline — all patterns integrated
import time
import math
import numpy as np
import re
import json
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

client = Anthropic()
HAIKU  = "claude-haiku-4-5-20251001"
SONNET = "claude-sonnet-4-20250514"

# ── Knowledge base ────────────────────────────────────────────────────────────
KB = [
    {"id": "K1", "text": "BGP AS number 65001. Hold timer 90s, keepalive 30s. ISP peer AS 65000 at 203.0.113.2."},
    {"id": "K2", "text": "BGP neighbor state IDLE: verify TCP port 179 reachable. Check AS numbers and routing table."},
    {"id": "K3", "text": "BGP hold timer mismatch fix: neighbor X timers keepalive holdtime. Both sides must match."},
    {"id": "K4", "text": "OSPF area 0 authentication MD5. Hello 10s dead 40s on broadcast. Router-ID 10.0.0.1."},
    {"id": "K5", "text": "OSPF EXSTART failure: MTU mismatch. Fix: ip ospf mtu-ignore on both interfaces."},
    {"id": "K6", "text": "Interface uplink MTU 9000 jumbo. Access MTU 1500 standard. Loopback 65535."},
    {"id": "K7", "text": "NTP primary 10.0.0.1 stratum-2. Backup 10.0.0.2. Configure: ntp server 10.0.0.1 prefer."},
    {"id": "K8", "text": "BGP maximum-prefix 1000 per neighbor. If hit, session drops. Raise limit or filter routes."},
]
kb_texts = [k["text"] for k in KB]

# ── Build retrieval index (hybrid) ────────────────────────────────────────────
vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
mat = vec.fit_transform(kb_texts)

def tokenize(t):
    return re.findall(r'[a-z0-9]+', t.lower())

def bm25_score(query, doc_idx, docs, k1=1.5, b=0.75):
    toks = [tokenize(d) for d in docs]
    N    = len(docs)
    avgdl = sum(len(t) for t in toks) / N
    df   = Counter(t for doc in toks for t in set(doc))
    q    = tokenize(query)
    doc  = toks[doc_idx]
    tf   = Counter(doc)
    dl   = len(doc)
    total = 0.0
    for term in q:
        if df.get(term, 0) == 0:
            continue
        idf = math.log((N - df[term] + 0.5) / (df[term] + 0.5) + 1)
        num = tf.get(term, 0) * (k1 + 1)
        den = tf.get(term, 0) + k1 * (1 - b + b * dl / avgdl)
        total += idf * num / den if den else 0
    return total

def hybrid_retrieve(query: str, top_k: int = 3) -> list:
    # Semantic
    q_vec  = vec.transform([query])
    sem    = cosine_similarity(q_vec, mat).flatten()
    sem_ranked = sorted(enumerate(sem), key=lambda x: x[1], reverse=True)[:6]
    # BM25
    bm25_ranked = sorted(
        [(i, bm25_score(query, i, kb_texts)) for i in range(len(KB))],
        key=lambda x: x[1], reverse=True
    )[:6]
    # RRF
    rrf = {}
    for rank, (idx, _) in enumerate(sem_ranked, 1):
        rrf[idx] = rrf.get(idx, 0) + 1/(60+rank)
    for rank, (idx, _) in enumerate(bm25_ranked, 1):
        rrf[idx] = rrf.get(idx, 0) + 1/(60+rank)
    top = sorted(rrf.items(), key=lambda x: x[1], reverse=True)[:top_k]
    return [KB[i]["text"] for i, _ in top]

# ── Semantic cache (simplified) ───────────────────────────────────────────────
class SimpleSemanticCache:
    def __init__(self, threshold=0.88):
        self.store   = {}
        self.vec     = None
        self.mat     = None
        self.keys    = []
        self.thresh  = threshold
        self.hits    = 0
        self.misses  = 0

    def lookup(self, query):
        if not self.keys:
            self.misses += 1
            return None
        v = TfidfVectorizer(ngram_range=(1,2), sublinear_tf=True)
        m = v.fit_transform(self.keys)
        q = v.transform([query])
        scores = cosine_similarity(q, m).flatten()
        best_idx = int(np.argmax(scores))
        if scores[best_idx] >= self.thresh:
            self.hits += 1
            return self.store[self.keys[best_idx]]
        self.misses += 1
        return None

    def put(self, query, answer):
        self.store[query] = answer
        self.keys = list(self.store.keys())

cache = SimpleSemanticCache(threshold=0.88)

# ── Metrics log ───────────────────────────────────────────────────────────────
metrics_log = []

# ── Full pipeline ─────────────────────────────────────────────────────────────
def production_rag(question: str, verbose: bool = True):
    t_pipeline_start = time.time()

    # Step 1: Semantic cache check
    cached = cache.lookup(question)
    if cached:
        if verbose:
            print(f"  ✅ CACHE HIT — returning stored answer")
            print(f"     Answer: {cached[:100]}...")
        metrics_log.append({"q": question[:40], "source": "cache", "cost": 0, "latency": 0.01})
        return cached

    # Step 2: Hybrid retrieval
    chunks = hybrid_retrieve(question, top_k=3)
    context = "
".join(f"[{i+1}] {c}" for i, c in enumerate(chunks))

    # Step 3: LLM cascade
    system = (
        "Answer using ONLY the provided context. Be concise. "
        "At the end write: CONFIDENCE: X (1-10)."
    )
    user_msg = f"Context:
{context}

Question: {question}"

    # Try Haiku first
    h_resp = client.messages.create(
        model=HAIKU, max_tokens=200, system=system,
        messages=[{"role": "user", "content": user_msg}],
    )
    h_raw = h_resp.content[0].text.strip()
    conf  = 5
    answer = h_raw
    model_used = "haiku"

    if "CONFIDENCE:" in h_raw:
        parts  = h_raw.rsplit("CONFIDENCE:", 1)
        answer = parts[0].strip()
        try:
            conf = int(parts[1].strip().split()[0])
        except Exception:
            pass

    if conf < 7:
        # Escalate
        s_resp = client.messages.create(
            model=SONNET, max_tokens=300, system=system,
            messages=[{"role": "user", "content": user_msg}],
        )
        s_raw  = s_resp.content[0].text.strip()
        answer = s_raw.rsplit("CONFIDENCE:", 1)[0].strip() if "CONFIDENCE:" in s_raw else s_raw
        model_used = "sonnet"

    # Step 4: Cache store
    cache.put(question, answer)

    # Step 5: Record metrics
    latency = time.time() - t_pipeline_start
    cost    = 0.001 if model_used == "haiku" else 0.009
    metrics_log.append({"q": question[:40], "source": model_used, "cost": cost, "latency": latency})

    if verbose:
        escalated = model_used == "sonnet"
        icon = "⬆️ ESCALATED→Sonnet" if escalated else "✅ Haiku"
        print(f"  {icon} | Conf: {conf}/10 | {latency:.2f}s")
        print(f"  Retrieved chunks: {len(chunks)}")
        print(f"  Answer: {answer[:120]}...")

    return answer

# ── Run the full pipeline ─────────────────────────────────────────────────────
questions = [
    "What is our BGP AS number?",
    "My BGP neighbor is stuck in IDLE state — what should I check?",
    "What is our BGP AS number?",           # repeat — should hit cache
    "What is the MTU on uplink interfaces?",
    "OSPF adjacency not forming past EXSTART — what is the fix?",
    "What is the AS number for our network?",   # paraphrase — should hit cache
]

print("="*65)
print("  FULL PRODUCTION PIPELINE DEMO")
print("="*65)

for q in questions:
    print(f"
  Query: "{q}"")
    production_rag(q, verbose=True)

# ── Final metrics report ──────────────────────────────────────────────────────
print(f"
{'='*65}")
print(f"  PRODUCTION METRICS REPORT")
print(f"{'='*65}")
print(f"  Cache hits:    {cache.hits} | Cache misses: {cache.misses} | "
      f"Hit rate: {cache.hits/(cache.hits+cache.misses):.0%}")
total_cost = sum(m["cost"] for m in metrics_log)
avg_latency = sum(m["latency"] for m in metrics_log) / len(metrics_log)
print(f"  Total cost:    ${total_cost:.4f} for {len(metrics_log)} queries")
print(f"  Avg latency:   {avg_latency:.2f}s")
print(f"  Est monthly:   ${total_cost/len(metrics_log)*2000*30:.2f} (2k queries/day)")


---
## Summary — What You Built

A complete production-ready RAG system with all the operational patterns:

### Demo progression:
1. **Semantic Cache** — TF-IDF embedding comparison, similarity threshold tuning, LRU+TTL eviction. At 40% hit rate → saves ~$400/month on 2000 queries/day
2. **LLM Cascade** — Haiku attempts first with self-assessed confidence, escalates to Sonnet only when needed. 65% Haiku rate → ~65% cost reduction
3. **RAG Evaluation** — LLM-as-Judge implements 3 RAGAS metrics: Faithfulness (most critical), Context Relevance, Answer Relevancy. Automated quality gate
4. **Streaming + Monitoring** — TTFT, throughput (tokens/sec), per-query cost tracking. Baseline for SLA measurement
5. **Full Pipeline** — Cache → Hybrid Retrieval → Cascade → Store → Metrics. The complete production loop

### The two biggest production problems and their solutions:

| Problem | Solution | Impact |
|---------|----------|--------|
| Cost ($300–1,200/month) | Semantic cache + LLM cascade | 50–70% reduction |
| Quality (hallucinations) | Faithfulness evaluation + grounding prompts | Catch < 0.8 faithfulness |

### Production checklist before going live:
- [ ] Semantic cache with tuned threshold (0.88–0.95 for networking domain)
- [ ] LLM cascade routing (simple→Haiku, complex→Sonnet)
- [ ] Automated faithfulness evaluation on 10% sample of queries
- [ ] TTFT and total latency monitoring with alerting
- [ ] TTL on cache entries (24–48h for documentation)
- [ ] Graceful degradation when retrieval fails (return best effort or escalate)

---
*Chapter 18 — RAG Production Patterns | AI for Networking and Security Engineers*
